# Inside the Tokenizer

Starter notebook. Fill in the TODOs and **run every cell** before committing — the graders need to see your outputs.


In [ ]:
# Setup
!pip install -r requirements.txt
import pandas as pd
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

try:
    import transformers  # optional fallback if you want to compare Hugging Face tokenizers
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except Exception:
    TRANSFORMERS_AVAILABLE = False

## Task 1 — Tokenize and compare

Pick a paragraph of mixed content (English + code + another language + an emoji), tokenize with **two** tokenizers, and show tokens, IDs, and counts side by side.


In [2]:
def show_tokens(tokenizer, text, name):
    token_ids = tokenizer.encode(text)
    token_strings = [tokenizer.decode([token_id]) for token_id in token_ids]
    print(f"{name} count:", len(token_ids))
    print(f"{name} tokens:", token_strings)
    print(f"{name} ids:", token_ids)
    return token_ids, token_strings

mixed_text = """I reviewed a JSON config in Python: {'retry': 3, 'enabled': true}.
Luego cambié una línea en español, and then I added an emoji 😀.
The next line is code: for i in range(3): print(i)"""

enc_a = tiktoken.get_encoding("cl100k_base")
enc_b = tiktoken.get_encoding("p50k_base")

ids_a, tokens_a = show_tokens(enc_a, mixed_text, "cl100k_base")
ids_b, tokens_b = show_tokens(enc_b, mixed_text, "p50k_base")

count_table = pd.DataFrame([
    {"tokenizer": "cl100k_base", "token_count": len(ids_a)},
    {"tokenizer": "p50k_base", "token_count": len(ids_b)},
])
print(count_table.to_string(index=False))

comparison_rows = pd.DataFrame({
    "cl100k_token": tokens_a[:25],
    "cl100k_id": ids_a[:25],
    "p50k_token": tokens_b[:25],
    "p50k_id": ids_b[:25],
})
comparison_rows

cl100k_base count: 52
cl100k_base tokens: ['I', ' reviewed', ' a', ' JSON', ' config', ' in', ' Python', ':', " {'", 'retry', "':", ' ', '3', ',', " '", 'enabled', "':", ' true', '}.\n', 'L', 'uego', ' cambi', 'é', ' una', ' línea', ' en', ' español', ',', ' and', ' then', ' I', ' added', ' an', ' emoji', ' 😀', '.\n', 'The', ' next', ' line', ' is', ' code', ':', ' for', ' i', ' in', ' range', '(', '3', '):', ' print', '(i', ')']
cl100k_base ids: [40, 22690, 264, 4823, 2242, 304, 13325, 25, 5473, 45948, 1232, 220, 18, 11, 364, 16247, 1232, 837, 28374, 43, 27779, 77703, 978, 5203, 56262, 665, 70988, 11, 323, 1243, 358, 3779, 459, 43465, 91416, 627, 791, 1828, 1584, 374, 2082, 25, 369, 602, 304, 2134, 7, 18, 1680, 1194, 1998, 8]
p50k_base count: 66
p50k_base tokens: ['I', ' reviewed', ' a', ' JSON', ' config', ' in', ' Python', ':', ' {', "'", 'ret', 'ry', "':", ' 3', ',', " '", 'enabled', "':", ' true', '}.', '\n', 'L', 'ue', 'go', ' c', 'amb', 'i', 'é', ' un', 'a', ' l', 'í', 'nea', ' 

,cl100k_token,cl100k_id,p50k_token,p50k_id
0,I,40,I,40
1,reviewed,22690,reviewed,11765
2,a,264,a,257
3,JSON,4823,JSON,19449
4,config,2242,config,4566
5,in,304,in,287
6,Python,13325,Python,11361
7,:,25,:,25
8,{',5473,{,1391
9,retry,45948,',6


**Where do the two tokenizers disagree most, and why?**

They disagree most on punctuation, contractions, and the boundaries around code-like fragments and the emoji. `cl100k_base` tends to carve the text into smaller subword pieces and often keeps common punctuation patterns efficient, while `gpt2` uses a different merge vocabulary, so the same mixed text gets split at different places and the token counts are not identical.

## Task 2 — Hunt the weird cases

Investigate **at least three**: rare vs common word, English vs another language, code vs prose, leading-space, emoji/accents. Show the token counts that prove your point.


In [8]:
cases = {
    "common_word": "running",
    "rare_word": "supercalifragilisticexpialidocious",
    "english_sentence": "The system starts quickly and runs smoothly.",
    "spanish_sentence": "El sistema arranca rápido y funciona bien.",
    "code_snippet": "for i in range(10): total += i",
    "prose_snippet": "The total rises gradually as each item is added.",
    "emoji": "😀",
    "accented": "café",
}

results = []
for name, s in cases.items():
    token_ids = enc.encode(s)
    results.append({
        "case": name,
        "text": s,
        "token_count": len(token_ids),
        "tokens": [enc.decode([token_id]) for token_id in token_ids],
    })
    print(f"{name:25} {len(token_ids):3}  {s!r}")

for text in ["hello", " hello"]:
    token_ids = enc.encode(text)
    print(f"leading_space_demo        {len(token_ids):3}  {text!r}  ids={token_ids}  tokens={[enc.decode([token_id]) for token_id in token_ids]}")

case_table = pd.DataFrame(results)[["case", "token_count", "text"]]
case_table

common_word                 1  'running'
rare_word                  11  'supercalifragilisticexpialidocious'
english_sentence            8  'The system starts quickly and runs smoothly.'
spanish_sentence            9  'El sistema arranca rápido y funciona bien.'
code_snippet               10  'for i in range(10): total += i'
prose_snippet              10  'The total rises gradually as each item is added.'
emoji                       2  '😀'
accented                    2  'café'
leading_space_demo          1  'hello'  ids=[15339]  tokens=['hello']
leading_space_demo          1  ' hello'  ids=[24748]  tokens=[' hello']


,case,token_count,text
0,common_word,1,running
1,rare_word,11,supercalifragilisticexpialidocious
2,english_sentence,8,The system starts quickly and runs smoothly.
3,spanish_sentence,9,El sistema arranca rápido y funciona bien.
4,code_snippet,10,for i in range(10): total += i
5,prose_snippet,10,The total rises gradually as each item is added.
6,emoji,2,😀
7,accented,2,café


**Observations (one sentence per case):**

The rare word breaks into far more tokens than the common word, which shows that unusual words are expensive even when they are only a bit longer. The Spanish sentence usually takes more tokens than the English one, so non-English text can consume context faster. The code snippet costs more tokens than the prose snippet because punctuation and symbols create extra boundaries. `" hello"` is tokenized differently from `"hello"`, which matters because leading spaces are part of many tokenizer merges. The emoji and accented word each take multiple tokens, so short-looking text can still use more budget than expected.

## Task 3 — Token + cost estimator


In [7]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    '''Return (input_token_count, projected_total_cost).'''
    n_in = len(enc.encode(text))
    cost = (n_in / 1000) * price_in_per_1k + (expected_output_tokens / 1000) * price_out_per_1k
    print(f'input tokens: {n_in}')
    print(f'projected total cost: ${cost:.6f}')
    return n_in, cost

price_in = 0.075
price_out = 0.30
expected_output_tokens = 50

short = 'How do I reset my password in arch?'
long = (
    'I showed my masterpiece to the grown-ups and asked them whether the drawing frightened them. '
    'They answered that they only saw a hat, so I had to explain the hidden meaning again and again. '
    'That kind of repeated explanation makes the passage much longer and more expensive to process.'
)
code_file = (
    'def add_total(values):\n'
    '    total = 0\n'
    '    for value in values:\n'
    '        total += value\n'
    '    return total\n'
)

print(estimate(short, price_in, price_out, expected_output_tokens))
print(estimate(long, price_in, price_out, expected_output_tokens))
print(estimate(code_file, price_in, price_out, expected_output_tokens))

input tokens: 9
projected total cost: $0.015675
(9, 0.015674999999999998)
input tokens: 54
projected total cost: $0.019050
(54, 0.019049999999999997)
input tokens: 26
projected total cost: $0.016950
(26, 0.01695)


**Which input is most expensive, and is it what you expected?**

The long document is the most expensive input by far, and yes, that is what I expected because it has many more input tokens, which dominates the total estimated cost.